In [ ]:
import numpy as np
%matplotlib notebook
import matplotlib.pyplot as plt
import ipywidgets as widgets
import serial
import time
from picoscope import ps5000a
import picosdk
from picosdk.discover import find_all_units
import serial.tools.list_ports as port_list
import chipwhisperer as cw

# Check if the Picoscope is connected
scopes = find_all_units()
for scope in scopes:
    print("Working with:")
    print(scope.info)
    scope.close()
ports = list(port_list.comports())
for p in ports:
    print (p)


**PicoScope setup**

In [ ]:
ps = ps5000a.PS5000a()
print("Found the following picoscope:")
print(ps.getAllUnitInfo())

# Since target runnning at 10 MHz and AES requires from trigger
obs_duration = 3.225E-6
# Sample at least 1260 points within that window
sampling_interval = obs_duration / 1260
# Configure timebase
(actualSamplingInterval, nSamples, maxSamples) = ps.setSamplingInterval(sampling_interval, obs_duration)
print("Nsamples : ", nSamples)
print("Sampling interval = %f us" % (actualSamplingInterval*nSamples*1E6))

# 50mV range on channel A, AC coupled, 20 MHz BW limit
ps.setChannel('A', 'AC', 0.05, 0.0, enabled=True, BWLimited=True)
# Channel B is trigger
ps.setChannel('B', 'DC', 10.0, 0.0, enabled=True)
ps.setSimpleTrigger('B', 2.0, 'Rising', timeout_ms=2000, enabled=True)

**CW305 setup**

In [ ]:
TARGET_PLATFORM = 'CW305_100t'
fpga_id = '100t'
# Programming the target with default AES bitstream
target = cw.target(None, cw.targets.CW305, fpga_id=fpga_id, force = True)

print(target.is_programmed())

target.vccint_set(1.0)
target.pll.pll_enable_set(True)         # enable PLL chip
target.pll.pll_outenable_set(False, 0)  # disable PLL 0
target.pll.pll_outenable_set(True, 1)   # enable PLL 1
target.pll.pll_outenable_set(False, 2)  # disable PLL 2
target.pll.pll_outfreq_set(10E6, 1)     # PLL1 frequency set to 10 MHz
# 1 ms is plenty idling time --> maybe not useful with picoscope (?) 
target.clksleeptime = 1 

**Capture loop**

In [ ]:
def pico_capture():
    # Arm the picoscope
    ps.runBlock()
    time.sleep(0.05)
    # Trigger the encryption on Target
    target.fpga_write(target.REG_USER_LED, [0x01])
    target.usb_trigger_toggle()
    ps.waitReady()
    # Capture the trace 
    data = ps.getDataV('A', nSamples, returnOverflow=False)
    return data

In [ ]:
from tqdm.notebook import tnrange
project_file = "sca_test_CW305.cwp"
project = cw.create_project(project_file, overwrite=True)

from Crypto.Cipher import AES
from chipwhisperer.common.traces import Trace

ktp = cw.ktp.Basic()
# Initialize cipher to verify DUT result:
key, text = ktp.next()
cipher = AES.new(bytes(key), AES.MODE_ECB)
print("Key: ", key)

N = 2000        # Number of traces
traces = []
textin = []
keys = []
data_mV = []

target.fpga_write(target.REG_CRYPT_KEY, key[::-1])

# Dummy capture call due to bug of using AC coupling
pico_capture()

for i in tnrange(N, desc='Capturing traces'):
    
    # Write plaintext to target
    inputtext = text[::-1]
    target.fpga_write(target.REG_CRYPT_TEXTIN, inputtext)

    # Capture the trace 
    data = pico_capture()
    traces.append(np.array(data))
    data_mV.append(np.array(data)*1E3)

    # Organize and store data
    response = target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)
    response = response[::-1]
    trace_i = Trace(np.array(data), text, response, key)
    project.traces.append(trace_i)
    
    # Sanity check with expected ciphertext
    #print("Response fpga: ",[ hex(el) for el in response])
    #print("Expected instead: ", [hex(el) for el in cipher.encrypt(bytes(text))])
    assert (list(response) == list(cipher.encrypt(bytes(text)))), "Incorrect encryption result!\nGot {}\nExp {}\n".format(list(response), list(text))
    
    key, text = ktp.next() 
    textin.append(text)
    keys.append(key)

project.save()
project.close()
target.dis()

**Plotting trace**

In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import CrosshairTool

output_notebook()
p = figure(plot_width=800)

xrange = range(len(traces[0]))
p.line(xrange, traces[4], line_color="red")
show(p)

In [ ]:
import chipwhisperer.analyzer as cwa
import chipwhisperer as cw
project_file = "sca_test_CW305.cwp"
project = cw.open_project(project_file)

In [ ]:
attack = cwa.cpa(project, cwa.leakage_models.last_round_state_diff)
cb = cwa.get_jupyter_callback(attack)
results = attack.run(cwa.get_jupyter_callback(attack, 10))

In [ ]:
from chipwhisperer.analyzer.attacks.models.aes.key_schedule import key_schedule_rounds
recv_lastroundkey = [kguess[0][0] for kguess in results.find_maximums()]
recv_key = key_schedule_rounds(recv_lastroundkey, 10, 0)
print("Recovered key: ", [hex(subkey) for subkey in recv_key])
key=list(project.keys[0])
assert (key == recv_key), "Failed to recover encryption key!\nGot {}\nExp {}\n".format(recv_key, key)
print("Key recovery : Success!")

Graphical Results

In [ ]:
import holoviews as hv
from holoviews.operation.datashader import datashade, shade, dynspread, rasterize
from holoviews.operation import decimate
import pandas as pd, numpy as np
plot_data = cwa.analyzer_plots(results)

In [ ]:

def byte_to_color(idx):
    return hv.Palette.colormaps['Category20'](idx/16.0)

a = []
b = []
hv.extension('bokeh')

# display the calculated correlation at each point in time for the coreect key
for key_byte in range(0, 16):
    data = plot_data.output_vs_time(key_byte)
    a.append(np.array(data[1]))
    b.append(np.array(data[2]))
    b.append(np.array(data[3]))

pda = pd.DataFrame(a).transpose().rename(str, axis='columns')
pdb = pd.DataFrame(b).transpose().rename(str, axis='columns')   
curve = hv.Curve(pdb['0'], "Sample").options(color='black')

for key_byte in range(1, 16):
    curve *= hv.Curve(pdb[str(key_byte)]).options(color=byte_to_color(key_byte))

for key_byte in range(0, 16):
    curve *= hv.Curve(pda[str(key_byte)]).options(color=byte_to_color(key_byte))
curve.opts(width=900, height=600)
#decimate(curve.opts(width=900, height=600)) 

In [ ]:
ret = plot_data.pge_vs_trace(0)
curve = hv.Curve((ret[0], ret[1], "Traces Used in Calculation", "Partial Guessing Entropy of Byte"))

for bnum in range(1,16):
    ret = plot_data.pge_vs_trace(bnum)
    curve *= hv.Curve((ret[0], ret[1])).opts(color=byte_to_color(bnum))
curve.opts(width=900, height=600)

In [ ]:
a = []
b = []
for bnum in range(0,16):
    data = plot_data.corr_vs_trace(bnum)
    best = [0]*len(data[1][0])
    for i in range(256):
        if i == key[bnum]:
            a.append(np.array(data[1][i]))
        else:
            if max(best) < max(data[1][i]): best=data[1][i]  
    b.append(np.array(best))
pda = pd.DataFrame(a).transpose().rename(str, axis='columns')   
pdb = pd.DataFrame(b).transpose().rename(str, axis='columns')
curve = hv.Curve(pdb['0'].tolist(), "Iteration Number", "Max Correlation").options(color='black')

for i in range(1, len(pdb.columns)):
    curve *= hv.Curve(pdb[str(i)]).options(color=byte_to_color(i))

for i in range(1, len(pda.columns)):
    curve *= hv.Curve(pda[str(i)]).options(color=byte_to_color(i))

curve.opts(width=900, height=600)   

In [ ]:
plot_data.plot_corr_vs_trace()

In [ ]:
leak_model = cwa.leakage_models.sbox_output
attack = cwa.cpa(project, leak_model)
results = attack.run(cwa.get_jupyter_callback(attack, 10))
recv_sboxkey = [kguess[0][0] for kguess in results.find_maximums()]
recv_key = key_schedule_rounds(recv_sboxkey, 10, 0)
print("Recovered key: ", [hex(subkey) for subkey in recv_key])
assert (key == recv_key), "Failed to recover encryption key!\nGot {}\nExp {}\n".format(recv_key, key)
print("Key recovery : Success!")

In [ ]:
plot_data = cwa.analyzer_plots(results)
plot_data.plot_pge_vs_trace()